<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/TelBot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # Cell 1: Install Libraries
# !pip install python-telegram-bot --upgrade --quiet
# !pip install yt-dlp --upgrade --quiet
# !pip install nest_asyncio --quiet # Added for Colab compatibility
# !apt-get install -y ffmpeg # yt-dlp often needs ffmpeg for merging video/audio

# print("Libraries installed successfully!")
# !pip install python-telegram-bot yt-dlp
# !apt install zip
# !pip install python-telegram-bot[ext] yt-dlp
# !apt install zip


In [ ]:
# ============================================================
# 1. Install Dependencies
# ============================================================
# Run this cell once in your environment (like Colab) if you haven't already
!pip install yt-dlp python-telegram-bot --quiet
# print("Dependencies installed.")

# ============================================================
# 2. Import Libraries
# ============================================================
import logging
import os
import shutil
import uuid
import asyncio
import re # For basic URL check
from telegram import Update, InputFile
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from telegram.constants import ParseMode, ChatAction # Import ChatAction
from telegram.error import TelegramError
from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError

# ============================================================
# 3. Configuration
# ============================================================
# IMPORTANT: Replace with your actual Bot Token
TELEGRAM_BOT_TOKEN = "7668113099:AAHwpP6FHOlifpSqhFZTj03Zc6r0k2HrR8o" # <--- PASTE YOUR ACTUAL BOT TOKEN HERE

# Directory to store downloads temporarily
DOWNLOAD_BASE_DIR = "/content/downloads" # Suitable for Colab, adjust if running elsewhere

# Telegram limits file uploads via the Bot API's sendDocument method to 2000MB (2GB).
# Set a slightly lower limit for safety margin for the final ZIP file.
# NOTE: While the official limit is high, Telegram might enforce lower limits (~50MB)
# for bot uploads via send_document unpredictably. We check against this high limit first.
MAX_TELEGRAM_ZIP_SIZE_MB = 1900
MAX_TELEGRAM_ZIP_SIZE_BYTES = MAX_TELEGRAM_ZIP_SIZE_MB * 1024 * 1024

# !! CRITICAL: Telegram limits files sent via sendVideo/sendAudio/sendPhoto by BOTS to 50MB !!
# This is the limit for individual video uploads using send_video.
MAX_BOT_INDIVIDUAL_UPLOAD_MB = 50
MAX_BOT_INDIVIDUAL_UPLOAD_BYTES = MAX_BOT_INDIVIDUAL_UPLOAD_MB * 1024 * 1024


# ============================================================
# 4. Logging Setup
# ============================================================
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", level=logging.INFO
)
# Reduce log spam from underlying libraries
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("telegram").setLevel(logging.WARNING)
logging.getLogger("telegram.ext").setLevel(logging.INFO) # Keep bot actions visible
logging.getLogger("yt_dlp").setLevel(logging.WARNING) # Reduce yt-dlp info logs unless debugging
# Get logger for this script
logger = logging.getLogger(__name__)

# ============================================================
# 5. Helper Functions
# ============================================================

async def cleanup(download_dir: str, zip_path: str = None):
    """
    Removes the specified download directory and zip file (if provided).
    Designed to be run in an executor thread to avoid blocking asyncio loop.
    """
    logger.info(f"Initiating cleanup for dir: {download_dir}, zip: {zip_path}")
    await asyncio.sleep(0.1) # Small yield to allow other tasks

    # Remove download directory
    if download_dir and os.path.exists(download_dir):
        try:
            if os.path.isdir(download_dir):
                shutil.rmtree(download_dir)
                logger.info(f"Successfully removed directory: {download_dir}")
            else:
                logger.warning(f"Path exists but is not a directory, attempting removal: {download_dir}")
                os.remove(download_dir) # Attempt removal if it's a file unexpectedly
                logger.info(f"Successfully removed file found at directory path: {download_dir}")
        except OSError as e:
            logger.error(f"OS Error removing directory {download_dir}: {e}", exc_info=False) # Less verbose logging for cleanup errors
        except Exception as e:
            logger.error(f"Unexpected error removing directory {download_dir}: {e}", exc_info=False)
    else:
        logger.debug(f"Download directory not found or not specified, skipping removal: {download_dir}") # Debug level for non-critical info

    # Remove zip file
    if zip_path and os.path.exists(zip_path):
        try:
            if os.path.isfile(zip_path):
                os.remove(zip_path)
                logger.info(f"Successfully removed zip file: {zip_path}")
            else:
                 logger.warning(f"Zip path exists but is not a file, skipping removal: {zip_path}")
        except OSError as e:
            logger.error(f"OS Error removing zip file {zip_path}: {e}", exc_info=False)
        except Exception as e:
            logger.error(f"Unexpected error removing zip file {zip_path}: {e}", exc_info=False)
    else:
        logger.debug(f"Zip file not found or not specified, skipping removal: {zip_path}") # Debug level

def is_youtube_url(url: str) -> bool:
    """Performs a basic regex check to see if a string looks like a YouTube URL."""
    if not isinstance(url, str):
        return False
    # Simple pattern: matches youtube.com/watch?, youtube.com/playlist?, youtu.be/, youtube.com/shorts/
    youtube_regex = re.compile(
        r'(https?://)?(www\.)?'
        r'(youtube|m\.youtube)\.(com|de|fr|co\.uk|ca|pl|etc)/(watch\?v=|playlist\?list=|shorts/)|' # Added more TLDs example
        r'(https?://)?(www\.)?'
        r'youtu\.be/'
    )
    return bool(youtube_regex.match(url))

# ============================================================
# 6. Core Download and Upload Logic
# ============================================================

async def download_and_send(update: Update, context: ContextTypes.DEFAULT_TYPE, url: str):
    """
    Handles the entire process: download, optional individual upload, zip, upload zip, cleanup.
    """
    if not update or not update.effective_chat or not update.message:
        logger.error("Update object is missing necessary attributes. Cannot process.")
        return

    chat_id = update.effective_chat.id
    message_id = update.message.message_id # Original user message ID
    processing_message = None # To store the bot's status message
    download_dir = None # Initialize download directory path
    zip_path = None     # Initialize zip file path
    request_id = str(uuid.uuid4()).split('-')[0] # Short unique ID for logging

    try:
        # 1. Send Initial Status Message
        try:
            processing_message = await context.bot.send_message(
                chat_id=chat_id,
                text="🔄 Processing your request...",
                reply_to_message_id=message_id
            )
            # Store message ID in context for potential use in error handler
            context.user_data['_processing_message_id'] = processing_message.message_id
        except TelegramError as e:
            logger.error(f"Failed to send initial processing message to chat {chat_id}: {e}")
            return
        except Exception as e:
            logger.error(f"Unexpected error sending initial message to chat {chat_id}: {e}", exc_info=True)
            return

        # 2. Create Unique Download Directory
        download_dir = os.path.join(DOWNLOAD_BASE_DIR, f"request_{request_id}")
        try:
            os.makedirs(download_dir, exist_ok=True)
            logger.info(f"[{request_id}] Created download directory: {download_dir}")
        except OSError as e:
             logger.error(f"[{request_id}] Failed to create download directory {download_dir}: {e}", exc_info=True)
             await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Internal Error: Could not create storage directory.")
             return

        # 3. Download using yt-dlp
        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="📥 Starting download...")
        logger.info(f"[{request_id}] Starting download for URL: {url}")

        ydl_opts = {
            'format': 'bestvideo[ext=mp4][height<=1080]+bestaudio[ext=m4a]/best[ext=mp4][height<=1080]/best[ext=mp4]/best',
            'outtmpl': os.path.join(download_dir, '%(title)s [%(id)s].%(ext)s'),
            'writesubtitles': True,
            'subtitleslangs': ['en', 'fa'], # English and Persian subtitles
            'subtitlesformat': 'srt',
            'writedescription': False,
            'writeinfojson': False,
            'writeannotations': False,
            'noplaylist': False,
            'ignoreerrors': True, # Continue playlist download even if one video fails
            'quiet': True,        # Suppress yt-dlp console output
            'noprogress': True,   # Don't show progress bars in console
            'postprocessors': [{
                'key': 'FFmpegVideoConvertor',
                'preferedformat': 'mp4', # Ensure output is MP4
            }],
            'concurrent_fragment_downloads': 5, # Speed up fragment downloads
            'retries': 10,                      # Retry downloads on network issues
            'socket_timeout': 60,               # Increased timeout
            'verbose': False, # Set to True for detailed yt-dlp debugging output
            # 'ffmpeg_location': '/path/to/ffmpeg', # Optional: specify if not in PATH
        }

        download_success = False
        try:
            loop = asyncio.get_running_loop()
            with YoutubeDL(ydl_opts) as ydl:
                 # Run blocking download in executor thread
                 await loop.run_in_executor(None, lambda: ydl.download([url]))
            logger.info(f"[{request_id}] yt-dlp download process completed for URL: {url}")
            download_success = True
        except DownloadError as e:
            # Try to extract a concise error message
            error_message = str(e)
            match = re.search(r'ERROR:\s*(.*?)(?:;|$)', error_message, re.IGNORECASE)
            if match:
                error_message = match.group(1).strip()
            else: # Fallback if regex fails
                error_message = error_message.split('\n')[0] # Get first line

            logger.error(f"[{request_id}] yt-dlp download error for {url}: {error_message}")
            await context.bot.edit_message_text(
                chat_id=chat_id,
                message_id=processing_message.message_id,
                text=f"❌ Download failed. Error: {error_message[:200]}{'...' if len(error_message)>200 else ''}"
            )
            # No return here, cleanup happens in finally
        except Exception as e:
            logger.error(f"[{request_id}] Unexpected error during download execution for {url}: {e}", exc_info=True)
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ An unexpected error occurred during download.")
            # No return here, cleanup happens in finally

        # 4. Check Download Results
        downloaded_files = []
        if download_success: # Only check directory if download didn't raise a caught exception
            try:
                if os.path.exists(download_dir) and os.path.isdir(download_dir):
                    downloaded_files = [f for f in os.listdir(download_dir) if os.path.isfile(os.path.join(download_dir, f))]
                else:
                     logger.warning(f"[{request_id}] Download directory {download_dir} missing or not a directory after supposedly successful download.")
            except OSError as e:
                logger.error(f"[{request_id}] Error listing files in {download_dir}: {e}", exc_info=True)
            except Exception as e:
                logger.error(f"[{request_id}] Unexpected error listing files in {download_dir}: {e}", exc_info=True)
            # Ensure downloaded_files is a list even if errors occurred
            if not isinstance(downloaded_files, list): downloaded_files = []


        if not downloaded_files:
            logger.warning(f"[{request_id}] No files found in {download_dir} after download attempt for {url}. URL might be invalid, private, empty, or skipped due to errors.")
            # Check if we already sent an error message from the download block
            try:
                current_message = await context.bot.get_message(chat_id=chat_id, message_id=processing_message.message_id)
                if "failed" not in current_message.text.lower() and "error" not in current_message.text.lower():
                     await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="⚠️ Download finished, but no files were obtained. Check URL or video/playlist status (private, deleted, empty?).")
            except TelegramError: # Message might have been deleted or inaccessible
                logger.warning(f"[{request_id}] Could not fetch status message {processing_message.message_id} to update 'no files' status.")
            return # Stop processing if nothing was downloaded

        logger.info(f"[{request_id}] Found {len(downloaded_files)} files in {download_dir}: {downloaded_files}")
        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"☑️ Download complete. Checking {len(downloaded_files)} file(s) for individual upload...")

        # 5. Send Individual MP4 Files (if within BOT size limit: 50MB)
        video_files_sent_count = 0
        files_to_zip = list(downloaded_files) # Start with all files
        large_files_skipped = [] # Files skipped due to size or upload errors

        for filename in sorted(downloaded_files):
            file_path = os.path.join(download_dir, filename)

            # Check if it's a video file intended for individual upload
            if filename.lower().endswith((".mp4", ".mkv", ".webm")): # Add other relevant video types if needed
                file_size = 0
                try:
                    if os.path.exists(file_path):
                         file_size = os.path.getsize(file_path)
                    else:
                         logger.warning(f"[{request_id}] File {file_path} listed but not found for size check. Skipping.")
                         large_files_skipped.append(f"{filename} (File Missing)")
                         continue # Skip this file
                except OSError as e:
                    logger.error(f"[{request_id}] Could not get size of {file_path}: {e}. Skipping direct upload.", exc_info=True)
                    large_files_skipped.append(f"{filename} (Error getting size)")
                    continue

                file_size_mb = file_size / (1024*1024)

                # --- !!! USE THE 50MB LIMIT FOR INDIVIDUAL BOT UPLOADS !!! ---
                if file_size > MAX_BOT_INDIVIDUAL_UPLOAD_BYTES:
                    logger.warning(f"[{request_id}] Skipping direct upload of video (>{MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB Bot limit): {filename} ({file_size_mb:.2f} MB)")
                    # Only add unique entries based on filename
                    if not any(f.startswith(filename) for f in large_files_skipped):
                        large_files_skipped.append(f"{filename} ({file_size_mb:.2f} MB)")
                    continue # Skip sending this large file individually, will be in ZIP

                # Try uploading the individual video file (it's <= 50MB)
                try:
                    await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"⬆️ Uploading video: {filename} ({file_size_mb:.2f} MB)...")
                    logger.info(f"[{request_id}] Attempting to send video (<= {MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB): {filename}")
                    await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.UPLOAD_VIDEO)

                    with open(file_path, 'rb') as video_file:
                        await context.bot.send_video(
                            chat_id=chat_id,
                            video=video_file,
                            caption=filename,
                            connect_timeout=60,
                            read_timeout=600,    # Long timeout for upload
                            write_timeout=600,   # Long timeout for upload
                            pool_timeout=600     # Long timeout for upload
                        )
                    logger.info(f"[{request_id}] Successfully sent video: {filename}")
                    video_files_sent_count += 1
                    # Keep it in the zip for completeness by default
                    await asyncio.sleep(3) # Prevent rate limiting

                except TelegramError as e:
                    logger.error(f"[{request_id}] Failed to send video {filename} (Size: {file_size_mb:.2f} MB): {e}", exc_info=True)
                    error_detail = f"Error: {e}"
                    reason = "Upload Error"
                    if "Request Entity Too Large" in str(e) or "FILE_TOO_LARGE" in str(e):
                         error_detail = f"Exceeded 50MB Bot API limit during upload attempt."
                         reason = "API limit error"
                    elif "wrong file identifier" in str(e).lower() or "invalid" in str(e).lower():
                         error_detail = "Invalid file format or Telegram couldn't process it."
                         reason = "Format/Processing error"

                    # Only add unique entries based on filename
                    if not any(f.startswith(filename) for f in large_files_skipped):
                         large_files_skipped.append(f"{filename} ({reason})")

                    # Notify user about the failure for this specific file
                    await context.bot.send_message(chat_id=chat_id, text=f"❌ Failed to send video '{filename}'. {error_detail} It will be included in the ZIP file.", reply_to_message_id=message_id)

                except FileNotFoundError:
                    logger.error(f"[{request_id}] File not found error during video upload: {file_path}", exc_info=True)
                    await context.bot.send_message(chat_id=chat_id, text=f"❌ Error: Could not find file '{filename}' for upload.", reply_to_message_id=message_id)
                    if not any(f.startswith(filename) for f in large_files_skipped):
                         large_files_skipped.append(f"{filename} (File Not Found)")

                except Exception as e:
                    logger.error(f"[{request_id}] Unexpected error sending video {filename}: {e}", exc_info=True)
                    await context.bot.send_message(chat_id=chat_id, text=f"❌ Unexpected error sending video '{filename}'. It will still be in the ZIP file.", reply_to_message_id=message_id)
                    if not any(f.startswith(filename) for f in large_files_skipped):
                        large_files_skipped.append(f"{filename} (Unexpected Error)")

            # else: # Handle non-video files (like .srt) - they remain in files_to_zip
            #     logger.debug(f"[{request_id}] File {filename} is not video/audio, skipping individual upload.")


        # Send summary about individual uploads if needed
        summary_message = ""
        if video_files_sent_count > 0:
            summary_message += f"✅ Sent {video_files_sent_count} video file(s) individually (<= {MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB). "
        if large_files_skipped:
             # Consolidate reasons
             size_skipped_count = sum(1 for f in large_files_skipped if "MB)" in f)
             error_skipped_count = len(large_files_skipped) - size_skipped_count
             reasons = []
             if size_skipped_count > 0: reasons.append(f"{size_skipped_count} video(s) >{MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB")
             if error_skipped_count > 0: reasons.append(f"{error_skipped_count} file(s) with upload/check errors")
             summary_message += f"⚠️ Skipped individual upload for {len(large_files_skipped)} file(s) ({', '.join(reasons)}). They will be in the ZIP."
        # If no videos sent and no files skipped, but there are files to zip (e.g., only subs)
        elif not summary_message and files_to_zip and video_files_sent_count == 0:
             summary_message = f"ℹ️ No video files were eligible for individual upload (check size > {MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB or download errors). Preparing ZIP for remaining files."

        if summary_message: # Only send if there's something to report
             await context.bot.send_message(chat_id=chat_id, text=summary_message.strip(), reply_to_message_id=message_id)


        # 6. Create and Send ZIP Archive (if there are files to zip)
        if not files_to_zip:
            logger.warning(f"[{request_id}] No files were available or left for zipping in {download_dir}.")
            final_text = "✅ Done."
            if video_files_sent_count == 0 and not large_files_skipped:
                 final_text = "⚠️ No files found or eligible for sending."
            elif video_files_sent_count > 0:
                 final_text = f"✅ Done. Sent {video_files_sent_count} video(s) individually. No other files to ZIP."
            elif not video_files_sent_count and large_files_skipped:
                 final_text = f"✅ Done. No videos sent individually (due to size/errors). No other non-video files to ZIP."

            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=final_text)
            return

        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"⚙️ Creating ZIP archive for {len(files_to_zip)} file(s)...")

        zip_filename_base = f"youtube_download_{request_id}"
        zip_path_base = os.path.join(DOWNLOAD_BASE_DIR, zip_filename_base) # Path without .zip

        try:
            logger.info(f"[{request_id}] Creating zip archive: {zip_path_base}.zip from directory {download_dir}")
            loop = asyncio.get_running_loop()
            # Run blocking zip operation in executor thread
            zip_path = await loop.run_in_executor(None, lambda: shutil.make_archive(zip_path_base, 'zip', download_dir))
            logger.info(f"[{request_id}] ZIP archive created successfully: {zip_path}")
        except Exception as e:
            logger.error(f"[{request_id}] Failed to create ZIP archive: {e}", exc_info=True)
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Failed to create ZIP file. Error: {e}")
            return # Cleanup in finally

        # Check zip size *before* attempting upload
        zip_size = 0
        try:
            if os.path.exists(zip_path):
                zip_size = os.path.getsize(zip_path)
            else:
                logger.error(f"[{request_id}] ZIP file {zip_path} not found after creation for size check.")
                await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Internal Error: ZIP file disappeared after creation.")
                return # Cleanup in finally
        except OSError as e:
             logger.error(f"[{request_id}] Could not get size of zip file {zip_path}: {e}", exc_info=True)
             await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Error checking ZIP file size after creation.")
             return # Cleanup in finally

        zip_size_mb = zip_size / (1024*1024)

        # --- !!! USE THE LARGE LIMIT (e.g., 1900MB) FOR THE ZIP CHECK !!! ---
        if zip_size > MAX_TELEGRAM_ZIP_SIZE_BYTES:
            logger.warning(f"[{request_id}] ZIP file {zip_path} ({zip_size_mb:.2f} MB) is too large (> {MAX_TELEGRAM_ZIP_SIZE_MB}MB) for Telegram upload.")
            final_message = f"⚠️ The final ZIP archive is too large ({zip_size_mb:.2f} MB, limit ~{MAX_TELEGRAM_ZIP_SIZE_MB}MB) to upload via Telegram."
            if video_files_sent_count > 0:
                final_message += f" {video_files_sent_count} smaller video(s) were sent individually."
            # Send this as a separate message
            await context.bot.send_message(chat_id=chat_id, text=final_message, reply_to_message_id=message_id)
            # Update the status message to reflect completion state
            status_text = f"✅ Done."
            if video_files_sent_count > 0: status_text += f" {video_files_sent_count} individual file(s) sent."
            status_text += " ZIP too large."
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=status_text.strip())
        else:
            # Try uploading the zip file (it's within the theoretical document size limit)
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"⬆️ Uploading ZIP archive ({zip_size_mb:.2f} MB)...")
            try:
                await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.UPLOAD_DOCUMENT)
                with open(zip_path, 'rb') as zip_file:
                    await context.bot.send_document(
                        chat_id=chat_id,
                        document=zip_file,
                        filename=os.path.basename(zip_path),
                        caption=f"📦 All downloaded files ({len(files_to_zip)} items). Includes videos >{MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB, subtitles, etc.",
                        connect_timeout=60,
                        read_timeout=1800, # Increased significantly for large ZIPs
                        write_timeout=1800,
                        pool_timeout=1800,
                        reply_to_message_id=message_id
                    )
                logger.info(f"[{request_id}] Successfully sent ZIP file: {zip_path}")
                final_status = "✅ Done! All files processed."
                if video_files_sent_count > 0:
                    final_status += f" {video_files_sent_count} video(s) sent individually."
                final_status += " Others are in the ZIP."
                await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=final_status.strip())

            except TelegramError as e:
                logger.error(f"[{request_id}] Failed to send ZIP file {zip_path} ({zip_size_mb:.2f} MB): {e}", exc_info=True)
                # Default error text
                error_text = f"❌ Failed to send ZIP file. Error: {e}"

                # --- !!! REFINED ERROR MESSAGE LOGIC FOR FILE_TOO_LARGE ON ZIP !!! ---
                if "Request Entity Too Large" in str(e) or "FILE_TOO_LARGE" in str(e):
                    # Acknowledge the discrepancy - Telegram rejected it unexpectedly low.
                    error_text = (f"❌ Failed to send ZIP: Telegram rejected the file ({zip_size_mb:.2f} MB) "
                                  f"due to its size limits, even though it was below the expected {MAX_TELEGRAM_ZIP_SIZE_MB}MB threshold. "
                                  f"Telegram's actual limit for bot document uploads might be lower (e.g., ~50MB) or vary.")
                # --- !!! END OF REFINED LOGIC !!! ---

                await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=error_text)
            except FileNotFoundError:
                 logger.error(f"[{request_id}] File not found error during ZIP upload: {zip_path}", exc_info=True)
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Error: Could not find ZIP file '{os.path.basename(zip_path)}' for upload.")
            except Exception as e:
                 logger.error(f"[{request_id}] Unexpected error sending ZIP {zip_path}: {e}", exc_info=True)
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="❌ Unexpected error sending ZIP file.")

    except TelegramError as te:
        # Catch Telegram errors potentially happening outside upload blocks (e.g., editing messages)
        logger.error(f"[{request_id or 'UNKNOWN'}] Telegram API error during processing for chat {chat_id}: {te}", exc_info=True)
        try:
            # Try to edit the status message if possible, otherwise send new
            error_text = f"❌ A Telegram error occurred: {te}. Please try again later."
            if processing_message:
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=error_text)
            # Avoid sending another message if editing failed, could spam user
        except Exception as final_e:
             logger.error(f"[{request_id or 'UNKNOWN'}] Failed to send final error message to chat {chat_id} after TelegramError: {final_e}")
    except Exception as e:
        # Catch-all for any other unexpected errors
        request_info = f"[{request_id or 'UNKNOWN'}]"
        logger.critical(f"{request_info} An critical overall error occurred in download_and_send for chat {chat_id}: {e}", exc_info=True)
        try:
            final_error_text = f"❌ A critical internal error occurred. Please report this if it persists.\nError reference: {request_id}"
            if processing_message:
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=final_error_text)
            else: # If status message failed initially
                 await context.bot.send_message(chat_id=chat_id, text=final_error_text, reply_to_message_id=message_id)
        except Exception as ie:
             logger.error(f"{request_info} Failed to send final critical error message to chat {chat_id}: {ie}")

    finally:
        # 7. Cleanup (Always runs)
        logger.info(f"[{request_id or 'UNKNOWN'}] Entering finally block for cleanup.")
        loop = asyncio.get_running_loop()
        # Run cleanup in executor as shutil.rmtree can be blocking
        await loop.run_in_executor(None, lambda: asyncio.run(cleanup(download_dir, zip_path)))
        logger.info(f"[{request_id or 'UNKNOWN'}] Cleanup task scheduled/completed.")

        # Clear processing message ID from context
        if '_processing_message_id' in context.user_data:
             del context.user_data['_processing_message_id']

        # Try deleting the intermediate "Processing..." message *if* it exists and *if* it wasn't updated to a final state.
        if processing_message:
            try:
                # Fetch current text to check its state
                current_message = await context.bot.get_message(chat_id=chat_id, message_id=processing_message.message_id)
                # Keywords indicating an intermediate/temporary state
                intermediate_keywords = ["processing", "starting", "downloading", "uploading", "checking", "creating zip"]
                is_intermediate = any(keyword in current_message.text.lower() for keyword in intermediate_keywords)
                # Keywords indicating a final state (don't delete)
                final_state_keywords = ["done", "error", "failed", "warning", "skipped", "too large", "✅", "❌", "⚠️"]
                is_final_state = any(keyword in current_message.text.lower() for keyword in final_state_keywords)

                if is_intermediate and not is_final_state:
                     await context.bot.delete_message(chat_id=chat_id, message_id=processing_message.message_id)
                     logger.info(f"[{request_id or 'UNKNOWN'}] Deleted intermediate status message {processing_message.message_id}.")
                else:
                     logger.info(f"[{request_id or 'UNKNOWN'}] Keeping final status message {processing_message.message_id}: '{current_message.text[:50]}...'")
            except TelegramError as e:
                if "message to delete not found" in str(e).lower():
                    logger.info(f"[{request_id or 'UNKNOWN'}] Processing message {processing_message.message_id} already deleted.")
                elif "message can't be deleted" in str(e).lower():
                     logger.warning(f"[{request_id or 'UNKNOWN'}] Bot may lack permission to delete messages in chat {chat_id}.")
                else: # Other errors like message not found, etc.
                    logger.warning(f"[{request_id or 'UNKNOWN'}] Could not delete or check processing message {processing_message.message_id}: {e}")
            except Exception as e:
                 logger.warning(f"[{request_id or 'UNKNOWN'}] Unexpected error during final message cleanup: {e}", exc_info=True)


# ============================================================
# 7. Telegram Bot Handlers
# ============================================================

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Sends a welcome message when the /start command is issued."""
    user = update.effective_user
    start_message = (
        "Hi{user_mention}! 👋 Send me a YouTube video or playlist URL.\n"
        "I'll download it (MP4, max 1080p, with EN/FA subs if available).\n"
        f"Videos ≤ {MAX_BOT_INDIVIDUAL_UPLOAD_MB}MB will be sent directly.\n"
        f"All files (incl. larger videos) will be zipped (max ~{MAX_TELEGRAM_ZIP_SIZE_MB}MB, actual limit may vary)."
    ).format(user_mention=f" {user.mention_html()}" if user else "")

    if user:
        await update.message.reply_html(start_message)
        logger.info(f"User {user.id} ({user.username or 'N/A'}) started the bot.")
    else:
        # Fallback for users without accessible user object (rare)
        await update.message.reply_text(start_message.replace(user.mention_html(), "")) # Remove HTML mention
        logger.info("Received /start from unknown user.")


async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Handles incoming text messages, checking for YouTube URLs."""
    # Ignore edits, channel posts, etc. Only handle direct messages containing text.
    if not update.message or not update.message.text or update.edited_message:
        logger.debug("Ignoring non-message update or edited message.")
        return

    message_text = update.message.text.strip() # Strip whitespace
    chat_id = update.effective_chat.id
    user = update.effective_user
    user_info = f"{user.id} ({user.username or 'N/A'})" if user else "Unknown User"

    logger.info(f"Received message from {user_info} in chat {chat_id}: '{message_text[:70]}{'...' if len(message_text)>70 else ''}'")

    # Basic check for http/https prefix
    if not message_text.lower().startswith(('http://', 'https://')):
         logger.info(f"Message from {user_info} does not start with http/https.")
         # Avoid replying if it's likely just chat text
         if len(message_text.split()) > 5 or len(message_text) > 100: # Heuristic for chat text vs bad link
              logger.debug("Ignoring likely chat message.")
              return
         await update.message.reply_text(
             "⚠️ Please send a valid YouTube video or playlist URL starting with `http://` or `https://`."
         )
         return

    if is_youtube_url(message_text):
        logger.info(f"Detected YouTube URL from {user_info}. Starting download process.")
        # Schedule the download task to run concurrently without blocking the handler
        asyncio.create_task(download_and_send(update, context, message_text))
    else:
        logger.info(f"Message from {user_info} is not a valid YouTube URL pattern.")
        await update.message.reply_text(
            "⚠️ That doesn't look like a valid YouTube video or playlist URL.\n"
            "Examples:\n"
            "<code>https://www.youtube.com/watch?v=...</code>\n"
            "<code>https://youtu.be/...</code>\n"
            "<code>https://www.youtube.com/playlist?list=...</code>\n"
            "<code>https://www.youtube.com/shorts/...</code>",
            parse_mode=ParseMode.HTML,
            disable_web_page_preview=True
        )

async def error_handler(update: object, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Logs errors caused by Updates and optionally notifies the user."""
    logger.error(f"Exception while handling an update: {context.error}", exc_info=context.error)

    # Ignore common, often transient or recoverable errors
    if isinstance(context.error, TelegramError) and (
        "Query is too old" in str(context.error) or
        "message is not modified" in str(context.error).lower() or
        "message to edit not found" in str(context.error).lower() or
        "message can't be deleted" in str(context.error).lower() or # Permissions issue
        "chat not found" in str(context.error).lower() or
        "bot was blocked by the user" in str(context.error).lower() or
        "user is deactivated" in str(context.error).lower() or
        "telegram.error.NetworkError" in str(type(context.error)) # Handle generic network errors silently sometimes
        # Add more specific error types to ignore if needed
        ):
        logger.warning(f"Ignoring common/expected TelegramError or NetworkError: {context.error}")
        return

    # Try to notify the user about unexpected errors
    chat_id = None
    user_message_id = None
    processing_message_id = context.user_data.get('_processing_message_id', None)

    if isinstance(update, Update):
        if update.effective_chat:
            chat_id = update.effective_chat.id
        if update.effective_message: # The message that *caused* the error (often the user's URL)
            user_message_id = update.effective_message.message_id

    if chat_id:
        try:
            error_text = "😥 Sorry, an unexpected error occurred while processing your request. " \
                         "The development team has been notified. Please try again later."

            # Determine best message to reply to: status message if known, else user message
            reply_id = processing_message_id or user_message_id

            # Check if the error is related to the processing message itself
            if processing_message_id and isinstance(context.error, TelegramError) and str(processing_message_id) in str(context.error):
                 # Don't try to reply to the message that caused the error
                 reply_id = user_message_id

            await context.bot.send_message(
                chat_id=chat_id,
                text=error_text,
                reply_to_message_id=reply_id # Reply to status msg or user msg
            )
        except Exception as e:
             # Log failure to notify user, but don't crash the handler itself
             logger.error(f"Failed to send error handler message to chat {chat_id}: {e}", exc_info=True)


# ============================================================
# 8. Main Function and Bot Lifecycle Management
# ============================================================

application = None # Global scope for shutdown access
stop_event = asyncio.Event() # Used to keep main alive

async def main() -> None:
    """Initializes the bot, sets up handlers, and starts polling."""
    global application

    if not TELEGRAM_BOT_TOKEN or "YOUR_BOT_TOKEN_HERE" in TELEGRAM_BOT_TOKEN or len(TELEGRAM_BOT_TOKEN.split(':')) != 2:
        logger.critical("FATAL: TELEGRAM_BOT_TOKEN is not set or is invalid! Exiting.")
        print("\n" + "="*60)
        print("  ERROR: Please set a valid Telegram Bot Token in the")
        print("         Configuration section (around line 25).")
        print("         A valid token looks like '1234567890:ABC...')")
        print("="*60 + "\n")
        return

    logger.info("Starting bot initialization...")

    # Consider using Persistence for user_data if needed across restarts
    # from telegram.ext import PicklePersistence
    # persistence = PicklePersistence(filepath="bot_persistence.pkl")
    # application = Application.builder().token(TELEGRAM_BOT_TOKEN).persistence(persistence)...

    application = (
        Application.builder()
        .token(TELEGRAM_BOT_TOKEN)
        # Set reasonable default timeouts
        .connect_timeout(30)
        .read_timeout(30)
        .write_timeout(30)
        # Pool timeout slightly higher for potentially reused connections
        .pool_timeout(60)
        # Timeouts specific to fetching updates (polling)
        .get_updates_connect_timeout(40)
        .get_updates_read_timeout(40)
        .get_updates_pool_timeout(70)
        .build()
    )

    # Ensure download base directory exists
    try:
        os.makedirs(DOWNLOAD_BASE_DIR, exist_ok=True)
        logger.info(f"Ensured download base directory exists: {DOWNLOAD_BASE_DIR}")
    except OSError as e:
         logger.error(f"Could not create/access download base directory {DOWNLOAD_BASE_DIR}: {e}. Downloads will likely fail.", exc_info=True)
         print(f"\nWARNING: Could not create download directory '{DOWNLOAD_BASE_DIR}'. Check permissions. Downloads might fail.\n")
         # Continue running, but log the warning

    # Register handlers
    application.add_handler(CommandHandler("start", start))
    # Handle text messages that are not commands and contain text
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND & filters.UpdateType.MESSAGE, handle_message))
    application.add_error_handler(error_handler)

    # Initialize application (fetches bot info, etc.)
    try:
        logger.info("Initializing Telegram application...")
        await application.initialize()
        bot_user = await application.bot.get_me()
        logger.info(f"Application initialized successfully for bot @{bot_user.username} (ID: {bot_user.id})")
    except TelegramError as e:
         logger.critical(f"CRITICAL: Failed to initialize Telegram application (TelegramError): {e}", exc_info=True)
         print(f"\n--- CRITICAL ERROR ---")
         print(f"Could not initialize the bot due to a Telegram API error.")
         print(f"Check your TELEGRAM_BOT_TOKEN and network connection to Telegram.")
         print(f"Error details: {e}")
         print(f"----------------------\n")
         return
    except Exception as e:
         logger.critical(f"CRITICAL: Failed to initialize Telegram application (Other Error): {e}", exc_info=True)
         print(f"\n--- CRITICAL ERROR ---")
         print(f"Could not initialize the bot due to an unexpected error.")
         print(f"Error details: {e}")
         print(f"----------------------\n")
         return

    # Start background tasks (like handler processing loops)
    try:
        logger.info("Starting application background tasks...")
        await application.start()
        logger.info("Application started successfully.")
    except Exception as e:
         logger.critical(f"CRITICAL: Failed to start Telegram application tasks: {e}", exc_info=True)
         print(f"\n--- CRITICAL ERROR ---")
         print(f"Could not start the bot's background tasks.")
         print(f"Error details: {e}")
         print(f"----------------------\n")
         # Attempt cleanup if initialization succeeded but start failed
         await application.shutdown()
         return

    # Start polling for updates
    try:
        logger.info("Starting updater polling...")
        await application.updater.start_polling(
            allowed_updates=Update.ALL_TYPES, # Process messages, commands, etc.
            drop_pending_updates=True, # Recommended to avoid processing old updates on restart
            timeout=30 # Poll timeout in seconds
        )
        logger.info("Updater started polling successfully.")
        print("\n" + "="*50)
        print(f"✅ Telegram Bot is UP and RUNNING!")
        print(f"   Bot Username: @{bot_user.username}")
        print(f"   Download Dir: {DOWNLOAD_BASE_DIR}")
        print(f"   Indiv. Upload Limit: {MAX_BOT_INDIVIDUAL_UPLOAD_MB} MB")
        print(f"   ZIP Upload Limit:    ~{MAX_TELEGRAM_ZIP_SIZE_MB} MB (Actual may vary)")
        print("\n   Send /start or a YouTube URL to your bot in Telegram.")
        print("   -> To STOP the bot: Interrupt the kernel (Ctrl+C in terminal,")
        print("      Runtime -> Interrupt execution or Stop button ⏹️ in Colab/Jupyter)")
        print("="*50 + "\n")

    except Exception as e:
         logger.critical(f"CRITICAL: Failed to start polling: {e}", exc_info=True)
         print(f"\n--- CRITICAL ERROR ---")
         print(f"Could not start polling for Telegram updates.")
         print(f"Error details: {e}")
         print(f"----------------------\n")
         # Attempt cleanup if start succeeded but polling failed
         await application.stop()
         await application.shutdown()
         return

    # Keep main alive until stop_event is set (e.g., by interruption via KeyboardInterrupt)
    await stop_event.wait()

    # --- Shutdown sequence starts here after stop_event is set ---
    logger.info("Stop event received or interruption detected. Initiating shutdown...")
    await shutdown_bot()


async def shutdown_bot():
    """Gracefully stops the bot's components."""
    global application
    if not application:
        logger.warning("Shutdown called, but application object not found.")
        print("Bot application instance not found, cannot perform shutdown steps.")
        return

    print("\n--- Shutting down the bot gracefully... ---")
    logger.info("--- Starting Graceful Bot Shutdown ---")

    # 1. Stop Polling first to prevent new updates coming in
    if application.updater and application.updater.running:
        logger.info("Stopping updater polling...")
        try:
            await application.updater.stop()
            logger.info("Updater stopped.")
            print("   Polling stopped.")
        except Exception as e:
            logger.error(f"Error stopping updater: {e}", exc_info=True)
            print(f"   Error stopping polling: {e}")
    else:
         logger.info("Updater not running or already stopped.")
         print("   Polling was not active.")

    # 2. Stop Application Background Tasks (allow handlers for ongoing updates to finish)
    if application.running:
         logger.info("Stopping application background tasks (handlers, etc.)...")
         try:
             await application.stop() # This waits for tasks to finish gracefully
             logger.info("Application tasks stopped.")
             print("   Application background tasks stopped.")
         except Exception as e:
             logger.error(f"Error stopping application tasks: {e}", exc_info=True)
             print(f"   Error stopping application tasks: {e}")
    else:
         logger.info("Application tasks not running or already stopped.")
         print("   Application tasks were not active.")

    # 3. Shutdown Application (closes network connections, cleans up resources)
    logger.info("Shutting down application (closing connections)...")
    try:
        await application.shutdown()
        logger.info("Application shutdown complete.")
        print("   Application connections closed.")
    except Exception as e:
        logger.error(f"Error during application shutdown: {e}", exc_info=True)
        print(f"   Error during final application shutdown: {e}")

    logger.info("--- Bot Shutdown Sequence Complete ---")
    print("--- Bot shutdown complete. ---")

# ============================================================
# 9. Execute the Main Function
# ============================================================

async def run_main_wrapper():
    """Wraps main() and handles graceful shutdown on interruption."""
    try:
        await main() # Run the main bot logic and polling loop
    except (KeyboardInterrupt, asyncio.CancelledError):
        logger.info("KeyboardInterrupt or CancelledError caught. Triggering shutdown.")
        print("\nInterruption detected. Shutting down...")
        if not stop_event.is_set():
             stop_event.set() # Signal main() loop (waiting on event) to exit gracefully
        # The shutdown logic is now called at the end of main() after stop_event.wait() finishes
    except Exception as e:
         # Catch any truly unexpected errors at the top level
         logger.critical(f"Fatal error during bot execution in run_main_wrapper: {e}", exc_info=True)
         print(f"\n--- A CRITICAL UNHANDLED ERROR OCCURRED ---")
         print(f"   Error: {e}")
         print(f"   Attempting emergency shutdown...")
         print(f"-------------------------------------------\n")
         if not stop_event.is_set():
             stop_event.set() # Try to signal shutdown
         # Attempt shutdown even on unexpected fatal errors
         # Check if application exists before calling shutdown directly here
         if application:
             await shutdown_bot() # Try to cleanup resources directly if main didn't reach its end
         else:
             logger.warning("Application object not available for emergency shutdown.")
    finally:
        logger.info("Exiting run_main_wrapper.")
        # Ensure event is set so script can potentially exit if run differently
        if not stop_event.is_set(): stop_event.set()


# --- Start Execution ---
if __name__ == "__main__":
    print("Script starting...")
    print("Make sure dependencies are installed (see top of file).")
    print(f"Using temporary download path: {DOWNLOAD_BASE_DIR}")

    try:
        # Check if an event loop is already running (common in Jupyter/Colab)
        loop = asyncio.get_running_loop()
        print("Detected running event loop (Jupyter/Colab environment). Creating task...")
        # Schedule the wrapper to run as a task on the existing loop
        # This allows the notebook cell to potentially finish while the bot runs in the background
        task = loop.create_task(run_main_wrapper())
        print("Bot task created. It will run in the background.")
        print("NOTE: The cell execution might finish, but the bot keeps running.")
        print("      Use the 'Interrupt execution' button/command (like Ctrl+C or Stop button ⏹️) to stop the bot.")
        # In a notebook, we don't block here with task.result() or await task
        # The stop_event.wait() inside main() keeps the actual bot logic alive.

    except RuntimeError as e:
        # If no loop is running (e.g., standard Python script)
        if "no running event loop" in str(e).lower():
            print("No running event loop found. Starting new one with asyncio.run()...")
            try:
                # Use asyncio.run() as the entry point for standard scripts
                asyncio.run(run_main_wrapper())
            except KeyboardInterrupt:
                # asyncio.run() handles KeyboardInterrupt and cancels tasks,
                # the cleanup is handled in run_main_wrapper's except block.
                print("\nKeyboardInterrupt caught during asyncio.run(). Shutdown initiated.")
        else:
            # Different RuntimeError, log it critically
            print(f"\nUnhandled RuntimeError during startup: {e}")
            logger.critical(f"Unhandled RuntimeError during startup: {e}", exc_info=True)

    except Exception as e:
        # Catch any other unexpected errors during startup attempt
        print(f"\nUnexpected error during startup: {e}")
        logger.critical(f"Unexpected error during startup: {e}", exc_info=True)

    # The final "Bot script finished execution." print might be misleading in notebooks
    # as the task runs in the background. Commenting it out or rephrasing needed.
    # print("Bot script finished execution.")
    print("Startup sequence complete. Bot task is running or startup failed.")